# LangChain en producción: concurrencia y operaciones asíncronas

**NOTA**: Se recomienda al lector leer la sección del anexo titulado [Programación asíncrona con asyncio en Python: una guía práctica](referencia1).

Cuando una aplicación LangChain crece en complejidad y empieza a manejar múltiples peticiones simultáneas, la ejecución secuencial (síncrona) se convierte rápidamente en el cuello de botella. Llamadas a modelos de lenguaje a través de la red, consultas a bases de datos vectoriales o integraciones con APIs externas son operaciones donde el programa pasa la mayor parte del tiempo **esperando**, no computando.

La programación asíncrona con `asyncio` permite que, durante esas esperas, el hilo siga atendiendo otras tareas. El resultado es una mejora sustancial en el rendimiento y la capacidad de respuesta de la aplicación.

## 1. Soporte asíncrono nativo en LangChain
```{index} invoke,ainvoke
```

LangChain está diseñado desde el principio para trabajar en entornos asíncronos. Casi todos sus componentes que implican I/O ofrecen una versión asíncrona siguiendo una convención sencilla:

| Método síncrono | Método asíncrono |
|---|---|
| `invoke()` | `ainvoke()` |
| `batch()` | `abatch()` |
| `stream()` | `astream()` |
| `transform_documents()` | `atransform_documents()` |

Para usar estos métodos solo necesitas escribir tu código con las palabras clave `async` y `await` de Python.

## 2. Conceptos esenciales de `asyncio`

Antes de ver ejemplos con LangChain, conviene tener claros cinco conceptos:

1. **`async def`** — declara una función como corrutina. Las corrutinas pueden pausarse y reanudarse.
2. **`await`** — pausa la corrutina actual hasta que la operación indicada termine. Mientras tanto, el bucle de eventos puede atender otras tareas.
3. **Bucle de eventos** — el núcleo de `asyncio`. Orquesta y distribuye la ejecución de todas las tareas asíncronas registradas.
4. **`asyncio.run(coroutine())`** — punto de entrada habitual para arrancar el bucle de eventos y ejecutar una corrutina de nivel superior.
5. **`asyncio.gather(*coroutines)`** — ejecuta múltiples corrutinas de forma concurrente y devuelve sus resultados como lista cuando todas han terminado.

## 3. De síncrono a asíncrono: usando `ainvoke`

Veamos cómo transformar una cadena síncrona básica en su versión asíncrona. La definición de la cadena con LCEL (`prompt | model | parser`) no cambia; solo cambia la forma de invocarla.

In [1]:
# Versión SÍNCRONA (bloquea el hilo mientras espera la respuesta)
# ---------------------------------------------------------------------------
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI
# from langchain_core.output_parsers import StrOutputParser
#
# plantilla = ChatPromptTemplate.from_template("Explica brevemente qué es {concepto}")
# modelo    = ChatOpenAI()
# parser    = StrOutputParser()
# cadena    = plantilla | modelo | parser
#
# resultado = cadena.invoke({"concepto": "la entropía en termodinámica"})
# print(resultado)
print("(Ejemplo síncrono comentado para no requerir API key)")

(Ejemplo síncrono comentado para no requerir API key)


In [2]:
import asyncio

# Versión ASÍNCRONA: misma cadena, distinto método de invocación
# ---------------------------------------------------------------------------
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_openai import ChatOpenAI
# from langchain_core.output_parsers import StrOutputParser
#
# plantilla = ChatPromptTemplate.from_template("Explica brevemente qué es {concepto}")
# modelo    = ChatOpenAI()
# parser    = StrOutputParser()
# cadena    = plantilla | modelo | parser

async def ejecutar_cadena_asincrona():
    print("Lanzando cadena asíncrona...")
    # resultado = await cadena.ainvoke({"concepto": "la entropía en termodinámica"})
    await asyncio.sleep(1)  # simula la espera de red
    resultado = "La entropía mide el grado de desorden de un sistema termodinámico."
    print(f"Resultado: {resultado}")

# En Jupyter usamos await directamente (el bucle ya está activo)
await ejecutar_cadena_asincrona()

Lanzando cadena asíncrona...
Resultado: La entropía mide el grado de desorden de un sistema termodinámico.


> **Nota sobre LCEL:** Si llamas `ainvoke` sobre una cadena LCEL, LangChain intentará invocar `ainvoke` en cada componente de la cadena. Si algún componente no tiene soporte asíncrono nativo, LCEL lo ejecutará en un `ThreadPoolExecutor` para no bloquear el bucle de eventos, aunque esto añade algo de sobrecarga. Para máximo rendimiento, elige componentes con soporte asíncrono nativo en todas las etapas que impliquen I/O.

## 4. Ejecución concurrente con `asyncio.gather`

El verdadero poder de la asincronía aparece cuando necesitas lanzar varias operaciones al mismo tiempo: consultar el LLM con distintos prompts, recuperar documentos de fuentes diferentes o validar múltiples entradas en paralelo.

`asyncio.gather` es la herramienta para esto. Recibe varias corrutinas y las ejecuta de forma concurrente, devolviendo los resultados en el mismo orden en que se pasaron.

In [3]:
import asyncio
import time

async def consultar_modelo(concepto: str) -> str:
    print(f"  → Iniciando consulta: '{concepto}'")
    await asyncio.sleep(1.5)  # simula latencia de red hacia el LLM
    # En producción sería: return await cadena.ainvoke({"concepto": concepto})
    respuesta = f"Resumen generado sobre '{concepto}'."
    print(f"  ✓ Consulta completada: '{concepto}'")
    return respuesta

async def ejecutar_consultas_concurrentes():
    conceptos = ["redes neuronales", "transformers", "aprendizaje por refuerzo"]
    
    inicio = time.time()

    # Creamos la lista de corrutinas y las lanzamos todas a la vez
    corrutinas = [consultar_modelo(c) for c in conceptos]
    resultados = await asyncio.gather(*corrutinas)

    fin = time.time()
    print(f"\nResultados obtenidos:")
    for r in resultados:
        print(f"  • {r}")
    print(f"\nTiempo total: {fin - inicio:.2f}s  "
          f"(secuencial hubiera tardado ~{1.5 * len(conceptos):.1f}s)")

await ejecutar_consultas_concurrentes()

  → Iniciando consulta: 'redes neuronales'
  → Iniciando consulta: 'transformers'
  → Iniciando consulta: 'aprendizaje por refuerzo'
  ✓ Consulta completada: 'redes neuronales'
  ✓ Consulta completada: 'transformers'
  ✓ Consulta completada: 'aprendizaje por refuerzo'

Resultados obtenidos:
  • Resumen generado sobre 'redes neuronales'.
  • Resumen generado sobre 'transformers'.
  • Resumen generado sobre 'aprendizaje por refuerzo'.

Tiempo total: 1.51s  (secuencial hubiera tardado ~4.5s)


Con ejecución secuencial, tres llamadas de 1,5 s cada una sumarían **4,5 s**. Con `asyncio.gather`, al superponerse las esperas de red, el tiempo total se acerca a **1,5 s** (la duración de la operación más larga), más un pequeño overhead.

## 5. Streaming asíncrono con `astream`

Los modelos de lenguaje generan texto token a token. Con `astream()` puedes procesar cada fragmento conforme llega, sin esperar a que la respuesta esté completa. Esto mejora enormemente la experiencia en aplicaciones conversacionales: el usuario empieza a leer la respuesta casi de inmediato.

In [4]:
import asyncio

async def fragmentos_simulados(tema: str):
    """Generador asíncrono que simula tokens llegando del LLM."""
    tokens = f"El concepto de '{tema}' es fundamental en el campo de la IA moderna.".split()
    for token in tokens:
        await asyncio.sleep(0.15)  # simula el intervalo entre tokens
        yield token + " "

async def mostrar_respuesta_en_streaming():
    tema = "atención multi-cabeza"
    print(f"Respuesta en streaming sobre '{tema}':\n")

    # En producción sería: async for chunk in cadena.astream({"concepto": tema}):
    async for fragmento in fragmentos_simulados(tema):
        print(fragmento, end="", flush=True)

    print("\n\n--- Streaming finalizado ---")

await mostrar_respuesta_en_streaming()

Respuesta en streaming sobre 'atención multi-cabeza':

El concepto de 'atención multi-cabeza' es fundamental en el campo de la IA moderna. 

--- Streaming finalizado ---


## 6. Procesamiento por lotes con `abatch`

`abatch()` es la versión asíncrona de `batch()`. Acepta múltiples entradas y las envía al componente subyacente de forma concurrente. Si el proveedor (modelo, API de embeddings, etc.) soporta peticiones en bloque reales, `abatch` puede ser más eficiente que varios `ainvoke` en paralelo porque reduce el número de llamadas a la red.

In [5]:
import asyncio
import time

# Simulamos un objeto que tiene abatch (como haría una cadena LCEL real)
class CadenaSimulada:
    async def abatch(self, entradas: list) -> list:
        """Procesa todas las entradas de forma concurrente."""
        async def procesar_una(entrada):
            await asyncio.sleep(1.2)  # simula llamada al LLM
            return f"Análisis completado: {entrada['tema']}"

        return await asyncio.gather(*[procesar_una(e) for e in entradas])

async def ejecutar_batch():
    temas = ["ética en IA", "computación cuántica", "arquitecturas serverless"]
    entradas = [{"tema": t} for t in temas]

    cadena = CadenaSimulada()
    inicio = time.time()

    print(f"Procesando lote de {len(entradas)} entradas...")
    resultados = await cadena.abatch(entradas)

    fin = time.time()
    print("\nResultados del batch:")
    for r in resultados:
        print(f"  • {r}")
    print(f"\nTiempo total: {fin - inicio:.2f}s")

await ejecutar_batch()

Procesando lote de 3 entradas...

Resultados del batch:
  • Análisis completado: ética en IA
  • Análisis completado: computación cuántica
  • Análisis completado: arquitecturas serverless

Tiempo total: 1.21s


> La mejora real de `abatch` frente a múltiples `ainvoke` depende de si el servicio subyacente (API del LLM, modelo de embeddings) optimiza genuinamente las peticiones en bloque. Si no lo hace, el comportamiento será equivalente a `asyncio.gather` con varios `ainvoke`.

## 7. Consideraciones para producción

### 7.1 Manejo de errores en `asyncio.gather`

Por defecto, si una corrutina lanza una excepción dentro de `gather`, esta se propaga de inmediato y puede cancelar las demás. Usa `return_exceptions=True` para capturar los errores como valores y seguir procesando el resto.

In [6]:
import asyncio

async def analizar_documento(doc_id: int) -> str:
    if doc_id == 2:
        raise ConnectionError(f"No se pudo acceder al documento #{doc_id}")
    await asyncio.sleep(0.8)
    return f"Documento #{doc_id} analizado correctamente."

async def procesar_varios_documentos():
    ids = [1, 2, 3, 4]

    resultados = await asyncio.gather(
        *[analizar_documento(i) for i in ids],
        return_exceptions=True  # los errores se devuelven como valores, no interrumpen
    )

    for idx, resultado in zip(ids, resultados):
        if isinstance(resultado, Exception):
            print(f"  ✗ Doc #{idx}: {type(resultado).__name__} — {resultado}")
        else:
            print(f"  ✓ Doc #{idx}: {resultado}")

await procesar_varios_documentos()

  ✓ Doc #1: Documento #1 analizado correctamente.
  ✗ Doc #2: ConnectionError — No se pudo acceder al documento #2
  ✓ Doc #3: Documento #3 analizado correctamente.
  ✓ Doc #4: Documento #4 analizado correctamente.


### 7.2 Limitar la concurrencia con `asyncio.Semaphore`

Lanzar demasiadas corrutinas a la vez puede saturar la API del LLM o agotar los recursos locales. Un `Semaphore` actúa como válvula: limita cuántas corrutinas pueden ejecutarse simultáneamente.

In [7]:
import asyncio

MAX_CONCURRENTES = 2  # máximo de llamadas simultáneas al LLM
semaforo = asyncio.Semaphore(MAX_CONCURRENTES)

async def llamar_llm_con_limite(pregunta: str) -> str:
    async with semaforo:  # espera hasta que haya un slot libre
        print(f"  Procesando: '{pregunta}'")
        await asyncio.sleep(1)  # simula llamada al LLM
        return f"Respuesta a '{pregunta}'"

async def gestionar_cola_preguntas():
    preguntas = [
        "¿Qué es RAG?",
        "¿Cómo funciona LCEL?",
        "¿Qué es un agente?",
        "¿Para qué sirven los embeddings?",
        "¿Qué es LangSmith?",
    ]

    print(f"Cola de {len(preguntas)} preguntas con límite de {MAX_CONCURRENTES} concurrentes:\n")
    resultados = await asyncio.gather(*[llamar_llm_con_limite(p) for p in preguntas])

    print("\nRespuestas obtenidas:")
    for r in resultados:
        print(f"  • {r}")

await gestionar_cola_preguntas()

Cola de 5 preguntas con límite de 2 concurrentes:

  Procesando: '¿Qué es RAG?'
  Procesando: '¿Cómo funciona LCEL?'
  Procesando: '¿Qué es un agente?'
  Procesando: '¿Para qué sirven los embeddings?'
  Procesando: '¿Qué es LangSmith?'

Respuestas obtenidas:
  • Respuesta a '¿Qué es RAG?'
  • Respuesta a '¿Cómo funciona LCEL?'
  • Respuesta a '¿Qué es un agente?'
  • Respuesta a '¿Para qué sirven los embeddings?'
  • Respuesta a '¿Qué es LangSmith?'


### 7.3 Integrar código síncrono bloqueante con `run_in_executor`

A veces necesitas llamar a una librería que solo tiene API síncrona (por ejemplo, lectura de archivos pesados, operaciones con bases de datos sin driver async, etc.). Para no bloquear el bucle de eventos, usa `loop.run_in_executor()`, que ejecuta la función en un hilo separado.

In [8]:
import asyncio
import time

def cargar_datos_sincrono(fuente: str) -> str:
    """Función síncrona bloqueante (simula lectura de disco o BD legacy)."""
    time.sleep(2)  # operación bloqueante
    return f"Datos cargados desde '{fuente}'"

async def cargar_sin_bloquear(fuente: str) -> str:
    bucle = asyncio.get_running_loop()
    # Ejecuta la función síncrona en un ThreadPoolExecutor
    resultado = await bucle.run_in_executor(None, cargar_datos_sincrono, fuente)
    return resultado

async def pipeline_mixto():
    print("Cargando datos de forma no bloqueante...")
    resultado = await cargar_sin_bloquear("base_de_datos_legacy")
    print(resultado)

await pipeline_mixto()

Cargando datos de forma no bloqueante...
Datos cargados desde 'base_de_datos_legacy'


---
## Resumen

| Patrón | Cuándo usarlo |
|---|---|
| `ainvoke` | Una sola llamada asíncrona a una cadena o componente |
| `asyncio.gather` | Varias operaciones independientes en paralelo |
| `astream` | Respuestas de LLMs token a token (chatbots, UIs reactivas) |
| `abatch` | Múltiples entradas cuando el backend soporta batching real |
| `Semaphore` | Limitar concurrencia para no saturar APIs externas |
| `run_in_executor` | Integrar código síncrono bloqueante sin detener el bucle |

Dominar estos patrones es imprescindible para construir aplicaciones LangChain que escalen de verdad en producción: mayor throughput, menor latencia percibida y un uso más eficiente de los recursos disponibles.